# 🎨 face2sketch — Phase 2: Finetune → Caricature (Kaggle)

**Load Phase 1 G weights, finetune on TwitterPicasso (184 pairs) for caricature style.**

### Prerequisites
- **GPU:** T4 x2 — enable in Notebook → Accelerator → GPU
- **Internet:** Enable in Notebook → Internet → On
- **Dataset:** Add `face2sketch` dataset as input (contains data.zip + pix2pix_best.pt)

### Strategy (v2)
- **λ_L1=200** — L1 does the heavy lifting (D dies early on 184 pairs)
- **50-epoch L1-only warmup** — build caricature structure first
- **LR=1e-4**, **Batch=32** — leverage T4 x2

In [ ]:
# @title 1. Clone Repo + Install
import os, sys, shutil
BASE = '/kaggle/working/face2sketch'
os.makedirs(BASE, exist_ok=True)
os.chdir('/kaggle/working')

!git clone https://github.com/weseegod/face2sketch.git {BASE} 2>/dev/null || true
os.chdir(BASE)
!git pull origin main

!pip install -q tqdm matplotlib pillow pyyaml einops
!mkdir -p checkpoints samples outputs

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print(f"BASE: {BASE}")
print('\n✅  Setup complete.')

# Stick to BASE for all later cells
os.environ['FACE2SKETCH_BASE'] = BASE

In [ ]:
# @title 2. Find Data + Checkpoint (recursive search)
import glob, os, shutil

BASE = os.environ.get('FACE2SKETCH_BASE', '/kaggle/working/face2sketch')
os.chdir(BASE)

# Search ALL of /kaggle/input/ for our files
data_zip = None
ckpt_src = None

for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == 'data.zip' and not data_zip:
            data_zip = os.path.join(root, f)
        if f == 'pix2pix_best.pt' and not ckpt_src:
            ckpt_src = os.path.join(root, f)

# ── Unzip data ──
if data_zip:
    print(f'📦  Unzipping {data_zip} ...')
    !unzip -o {data_zip} -d {BASE}/ 2>&1 | tail -3
    
    # Fix nested data/data/ if it happened
    nested = os.path.join(BASE, 'data', 'data')
    if os.path.isdir(nested):
        print('⚠️   Found data/data/ nesting — flattening ...')
        for sub in os.listdir(nested):
            src = os.path.join(nested, sub)
            dst = os.path.join(BASE, 'data', sub)
            if not os.path.exists(dst):
                shutil.move(src, dst)
        shutil.rmtree(nested)
        print('    ✅  Flattened: data/data/ → data/')
else:
    print('❌  data.zip not found in /kaggle/input/')

# ── Copy checkpoint ──
os.makedirs(f'{BASE}/checkpoints', exist_ok=True)
if ckpt_src:
    shutil.copy(ckpt_src, f'{BASE}/checkpoints/pix2pix_best.pt')
    print(f'📂  Checkpoint: pix2pix_best.pt ✅')
else:
    print('❌  pix2pix_best.pt not found in /kaggle/input/')

# ── Verify ──
ft_photos = len(os.listdir(f'{BASE}/data/finetune/photos')) if os.path.isdir(f'{BASE}/data/finetune/photos') else 0
ft_sketch = len(os.listdir(f'{BASE}/data/finetune/sketches')) if os.path.isdir(f'{BASE}/data/finetune/sketches') else 0
ds_photos = len(os.listdir(f'{BASE}/data/dataset/photos')) if os.path.isdir(f'{BASE}/data/dataset/photos') else 0
ckpt_ok = os.path.exists(f'{BASE}/checkpoints/pix2pix_best.pt')

print(f'\n{"─"*40}')
print(f'  data/dataset/  : {ds_photos//2} pairs')
print(f'  data/finetune/ : {min(ft_photos, ft_sketch)} pairs')
print(f'  Checkpoint     : {"✅" if ckpt_ok else "❌ MISSING"}')
print(f'{"─"*40}\n')

if not ckpt_ok or min(ft_photos, ft_sketch) == 0:
    print('⛔  Cannot proceed. Fix dataset and re-run cells 1-2.')
    raise SystemExit(1)

In [ ]:
# @title 3. Finetune — 100 epochs (~30 min T4 x2)
import os

BASE = os.environ.get('FACE2SKETCH_BASE', '/kaggle/working/face2sketch')
os.chdir(BASE)

assert torch.cuda.is_available(), "⚠️  Enable GPU in Notebook settings"
assert os.path.exists('checkpoints/pix2pix_best.pt'), "❌ Missing checkpoint — re-run cell 2"
assert os.path.isdir('data/finetune'), "❌ Missing finetune data — re-run cell 2"

print("🎯  Finetuning Phase 1 G on TwitterPicasso caricatures")
print("    λ_L1=200 | 50-epoch L1 warmup | LR=1e-4 | Batch=32")
print(f"    Dataset: 184 pairs  |  Epochs: 100\n")

!python src/train.py --mode finetune \
    --config configs/pix2pix_phase2.yaml \
    --finetune checkpoints/pix2pix_best.pt \
    --device cuda \
    --batch-size 32 \
    --name phase2_

print("\n✅  Finetuning complete → checkpoints/phase2_best.pt")

In [ ]:
# @title 4. Evaluate — Phase 2 Gate Check
import os, glob

BASE = os.environ.get('FACE2SKETCH_BASE', '/kaggle/working/face2sketch')
os.chdir(BASE)

ckpt = 'checkpoints/phase2_best.pt'
if not os.path.exists(ckpt):
    candidates = sorted(glob.glob('checkpoints/phase2_*.pt'), reverse=True)
    for c in candidates:
        if os.path.getsize(c) > 100000:
            ckpt = c; break

if not os.path.exists(ckpt):
    print('❌  No Phase 2 checkpoint found.')
elif not os.path.exists('data/test/photos'):
    print('⚠️  No test data found.')
else:
    !python src/evaluate.py --checkpoint {ckpt} --mode quick --device cuda

    print('\n📊  Comparing Phase 1 vs Phase 2 outputs...')
    p1_ckpt = 'checkpoints/pix2pix_best.pt'
    if os.path.exists(p1_ckpt):
        !python src/evaluate.py --checkpoint {p1_ckpt} \
            --checkpoint2 {ckpt} --mode compare --device cuda
    print('   ✅  Comparison: outputs/phase1_vs_phase2.png')

In [ ]:
# @title 5. Save Results to Kaggle Output
import os, shutil

BASE = os.environ.get('FACE2SKETCH_BASE', '/kaggle/working/face2sketch')
OUT  = '/kaggle/working'

for item in ['checkpoints', 'samples', 'outputs']:
    src = os.path.join(BASE, item)
    dst = os.path.join(OUT, item)
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"✅  Copied {item}/ → /kaggle/working/{item}/")

print("\n--- Kaggle Output (/kaggle/working/) ---")
for root, dirs, files in os.walk(OUT):
    for f in sorted(files):
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        rel = os.path.relpath(path, OUT)
        label = f"{size/(1024*1024):.0f}MB" if size > 1024*1024 else f"{size/1024:.0f}KB"
        print(f"  {rel}  ({label})")

print("\n🎉  All done! Download from Kaggle Output tab.")